# 00 — Business Understanding

This notebook frames the business problem, defines success criteria, and lays out the analytical plan before touching any data.

---
**Reference:** *Hands-On Machine Learning with Scikit-Learn and PyTorch* — Aurélien Géron (2025), Chapter 2: *End-to-End Machine Learning Projects*.

> *"Before writing a single line of code, you should clearly define the problem you are trying to solve."* — Géron, Ch.2

## 1. Business Problem

An e-commerce operation sells electronics across hundreds of product SKUs. The pricing team applies blanket discount rules (e.g., −10% on all products on Black Friday) without knowing:

- Which products will actually increase demand enough to compensate for lower prices.
- Which products are inelastic and will simply lose revenue.
- Which products are substitutes (raising one's price boosts the other's demand).

**Core question:** *By how much does demand for product X change when its price changes by 1%?*

This is the classic **price elasticity of demand** problem from microeconomics:

$$\varepsilon = \frac{\%\Delta Q}{\%\Delta P}$$

- **ε < −1:** elastic — discount increases total revenue.
- **−1 < ε < 0:** inelastic — discount reduces total revenue.
- **ε > 0:** Giffen/prestige good — demand rises with price (rare).

## 2. Framing the ML Task

Following Géron's checklist for framing the problem:

| Dimension | Decision |
|-----------|----------|
| **Supervised / Unsupervised** | Supervised (we observe both price and demand) |
| **Task type** | Regression (estimate a continuous coefficient per product) |
| **Algorithm** | OLS Log-log regression (simple, interpretable, correct for elasticity) |
| **Output** | One elasticity coefficient β per product with confidence interval |
| **Batch / Online** | Batch — re-run when new price/demand data arrives |
| **Performance measure** | R², p-value, CI width — not predictive accuracy |

**Why log-log?** Taking logs of both price and demand linearises the elasticity relationship:

$$\ln(\text{demand}) = \alpha + \beta \cdot \ln(\text{price}) + \varepsilon$$

The slope β is directly the elasticity coefficient — interpretable, unit-free.

## 3. Success Criteria

| Criterion | Target |
|-----------|--------|
| Products with estimable elasticity | ≥ 300 products |
| Statistically significant estimates (p < 0.05) | ≥ 30% of products |
| High-quality fit (R² ≥ 0.10) | ≥ 50% of filtered products |
| 95% CI width | < 2.0 for reliable estimates |
| Dashboard response time | < 3s for full simulation |

**Exclusion rule:** Products with fewer than 12 observations or fewer than 3 distinct price points are excluded — insufficient price variation to fit a regression.

## 4. Analytical Plan

1. **Data Understanding** (`01_`) — Validate schema, granularity, missing values, price and demand ranges.
2. **Exploratory Analysis** (`02_`) — Price distributions, temporal trends, discount patterns, product clusters.
3. **Feature Engineering** (`03_`) — Log transformation, discount rate, weekly aggregation, sklearn Pipeline.
4. **Modeling** (`04_`) — OLS per product, R² filter, confidence intervals, bootstrap, business translation.
5. **Deployment** (`05_`) — Streamlit dashboard, pipeline automation, CSV output.

## 5. Hypotheses to Validate

| # | Hypothesis | Expected Direction |
|---|------------|--------------------|
| H1 | Electronics are price-inelastic on average | Most β between −1 and 0 |
| H2 | Higher-discount products have more stable estimates | Lower SE for frequently discounted SKUs |
| H3 | Products in the same category share elasticity magnitude | Category-level median ε ≠ overall median |
| H4 | Revenue simulations show a non-linear sweet spot for discounts | Optimal discount ≠ maximum discount |

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from price_elasticity.config import load_config

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Project:', config['project']['name'])
print('Business goal:', config['project']['business_goal'])
print('Primary metric:', config['project']['primary_metric'])

In [ ]:
import pprint
pprint.pprint(config)